# Mamba-2 generation — `state-spaces/mamba2-130m`

Load the HF checkpoint into our `Mamba2Model`, generate with our
state-carrying `generate_native`, and compare greedy continuations to
the `mamba_ssm` reference model.

Expected: token-exact (or nearly) match on short prompts.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent / 'src'))

import torch
from transformers import AutoTokenizer
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

from mamba_minimal.model import load_model
from mamba_minimal.generate import generate_native

device = 'cuda'
dtype = torch.float32
name = 'state-spaces/mamba2-130m'

tok = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')
ours = load_model(name, device=device, dtype=dtype)
ref = MambaLMHeadModel.from_pretrained(name, device=device, dtype=dtype).eval()
print('ours:', type(ours).__name__, '| ref:', type(ref).__name__)

/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/triton/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custo

/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/triton/ssd_combined.py:757: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/ops/triton/ssd_combined.py:835: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

/home/ubuntu/miniconda3/envs/minimamba/lib/python3.11/site-packages/mamba_ssm/utils/hf.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(resolved_archiv

ours: Mamba2Model | ref: MambaLMHeadModel


In [2]:
prompts = [
    'Mamba-2 is useful because',
    'The capital of France is',
    'State space models can',
]
max_new = 24

for p in prompts:
    ids = tok(p, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        out_ours = generate_native(ours, ids, max_new_tokens=max_new, temperature=0.0)
        out_ref = ref.generate(ids, max_length=ids.shape[1] + max_new)
    print('PROMPT:', repr(p))
    print('  ours:', repr(tok.decode(out_ours[0])))
    print('  ref :', repr(tok.decode(out_ref[0])))
    match = (out_ours[0, :out_ref.shape[1]] == out_ref[0]).float().mean().item()
    print(f'  token agreement: {match:.3f}')
    print()

PROMPT: 'Mamba-2 is useful because'
  ours: 'Mamba-2 is useful because it is easy to use and it is easy to use.\n\nThe Mamba-2 is a great tool for'
  ref : 'Mamba-2 is useful because it is a very simple and easy to use tool. It is a very simple tool to create a Mamba-2'
  token agreement: 0.323



PROMPT: 'The capital of France is'
  ours: 'The capital of France is a place where you can find a lot of things to do. It’s a place where you can go to the'
  ref : 'The capital of France is a city of the French people, and the capital of the French Republic. It is the capital of the Republic of France'
  token agreement: 0.207



PROMPT: 'State space models can'
  ours: 'State space models can be used to model the behavior of a system of interacting particles. In particular, the model can be used to model the'
  ref : 'State space models can be used to model the dynamics of a system of interacting particles. The dynamics of a system of interacting particles is described by'
  token agreement: 0.571



In [3]:
# Spot-check: same seed → same sampled continuation between our path and a full-prefill path.
ids = tok('Long-range state space duality', return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    out = generate_native(ours, ids, max_new_tokens=32, temperature=0.7, top_k=50, seed=42)
print(tok.decode(out[0]))

Long-range state space duality for the KdV equation and the Wigner-Eckart equation for the Gaussian chaos.
In this paper, we consider the KdV equation
